# 06 · The KV cache and inference speed

After this notebook you can explain prefill vs decode, implement generation with and without a KV cache (and prove both produce identical tokens), compute KV-cache memory for any model config, and explain the serving tricks built on top of it: batching, continuous batching, PagedAttention, prefix caching and speculative decoding - plus the two latency numbers users feel, TTFT and TPOT.

**Time:** ~35 min · **Runs:** offline on CPU in about a minute · **Needs:** [03 · Attention](03_attention_from_scratch.ipynb), [04 · Tiny GPT](04_transformer_block_and_tiny_gpt.ipynb)

## Why this matters in interviews

- "Explain the KV cache - what it stores, why it works, what it costs" is the most common inference question; the memory formula is expected from memory.
- Serving design questions (GPU sizing, max batch, long context, cost per token) are all KV-cache arithmetic.
- Knowing which optimisation moves which number (TTFT vs TPOT vs throughput) is what separates a good answer from buzzwords.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `in02` | Explain the KV cache — what it stores, why it works, and what it costs. |
| `tr14` | What is the KV cache, and why does it exist? |
| `tr16` | Explain the difference between prefill and decode. |
| `tr17` | Why is the first token so much slower than the rest? |
| `in35` | Explain the arithmetic intensity argument for why decode is memory-bound. |
| `in11` | What is the difference between latency and throughput here, and why do they fight? |
| `in04` | What is continuous batching, and why does it matter so much? |
| `in12` | What is PagedAttention? |
| `in05` | How does speculative decoding work, and when does it lose? |

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
rng = np.random.default_rng(0)
torch.set_num_threads(1)          # tiny matmuls: one thread is fastest and immune to a busy CPU
pd.set_option("display.width", 120)
print("torch", torch.__version__)

## 1. Prefill and decode

**In plain English:** answering a prompt happens in two phases.

1. **Prefill:** the model reads the whole prompt in *one* forward pass (all positions in parallel) and produces the first output token. Lots of arithmetic, done in parallel -> *compute-bound*. Its duration is most of the **time to first token (TTFT)**.
2. **Decode:** then it generates one token per forward pass, each depending on the previous one. Little arithmetic per step, but every step must read all the weights (and the cache) from memory -> *memory-bandwidth-bound*. Its per-step duration is the **time per output token (TPOT)**.

Without a cache, each decode step would re-run the whole sequence. The causal mask (notebook 03) means the keys and values of past tokens **never change** once computed, so we store them - the **KV cache** - and each new step only computes K and V for the one new token.

```
no cache : step t processes tokens 1..t        -> work per step grows with t, total O(n²)
KV cache : step t processes token t only, and reads cached K,V for 1..t-1 -> work per step ~constant
```

## 2. A small transformer that can run with or without a cache

Random weights are fine - we are measuring mechanics, not quality.

In [ ]:
class CachedAttention(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.h, self.qkv, self.proj = n_heads, nn.Linear(d, 3 * d), nn.Linear(d, d)

    def forward(self, x, cache=None):
        B, T, d = x.shape
        q, k, v = self.qkv(x).split(d, dim=-1)
        q, k, v = (t.view(B, T, self.h, d // self.h).transpose(1, 2) for t in (q, k, v))   # (B, h, T, dh)
        if cache is not None:
            if "k" in cache:                                    # append the new keys/values to the cache
                k, v = torch.cat([cache["k"], k], dim=2), torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v
        S = k.shape[2]                                          # total length (cached + new)
        allowed = torch.ones(T, S, dtype=torch.bool).tril(diagonal=S - T)   # new query i sees keys <= its position
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed)
        return self.proj(y.transpose(1, 2).reshape(B, T, d))

class Block(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.ln1, self.attn, self.ln2 = nn.LayerNorm(d), CachedAttention(d, n_heads), nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(self, x, cache=None):
        x = x + self.attn(self.ln1(x), cache)
        return x + self.mlp(self.ln2(x))

class SmallLM(nn.Module):
    def __init__(self, vocab=256, d=128, n_layers=4, n_heads=4, max_len=1024):
        super().__init__()
        self.tok, self.pos = nn.Embedding(vocab, d), nn.Embedding(max_len, d)
        self.blocks = nn.ModuleList([Block(d, n_heads) for _ in range(n_layers)])
        self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, vocab, bias=False)
    def forward(self, idx, caches=None, start_pos=0):
        x = self.tok(idx) + self.pos(torch.arange(start_pos, start_pos + idx.shape[1]))
        for i, blk in enumerate(self.blocks):
            x = blk(x, None if caches is None else caches[i])
        return self.head(self.ln(x))

model = SmallLM().eval()
print(f"parameters: {sum(p.numel() for p in model.parameters()):,} | 4 layers, d=128, 4 heads")

In [ ]:
@torch.no_grad()
def generate_no_cache(model, prompt, n_new):
    idx, times, step_logits = prompt.clone(), [], []
    for _ in range(n_new):
        t0 = time.perf_counter()
        logits = model(idx)[:, -1, :]                      # re-run the WHOLE sequence every step
        nxt = logits.argmax(-1, keepdim=True)
        times.append(time.perf_counter() - t0)
        step_logits.append(logits); idx = torch.cat([idx, nxt], dim=1)
    return idx, times, step_logits

@torch.no_grad()
def generate_with_cache(model, prompt, n_new):
    caches = [{} for _ in model.blocks]
    times, step_logits = [], []
    t0 = time.perf_counter()
    logits = model(prompt, caches, start_pos=0)[:, -1, :]  # prefill: the whole prompt in one pass
    idx = prompt.clone()
    for step in range(n_new):
        if step > 0:
            t0 = time.perf_counter()
            logits = model(idx[:, -1:], caches, start_pos=idx.shape[1] - 1)[:, -1, :]   # decode: ONE token
        nxt = logits.argmax(-1, keepdim=True)
        times.append(time.perf_counter() - t0)
        step_logits.append(logits); idx = torch.cat([idx, nxt], dim=1)
    return idx, times, step_logits, caches

prompt = torch.randint(0, 256, (1, 32), generator=torch.Generator().manual_seed(0))
N_NEW = 200
out_nc, t_nc, logits_nc = generate_no_cache(model, prompt, N_NEW)
out_c, t_c, logits_c, caches = generate_with_cache(model, prompt, N_NEW)
assert torch.equal(out_nc, out_c), "the KV cache must not change a single token"
assert all(torch.allclose(a, b, atol=1e-4) for a, b in zip(logits_nc, logits_c))
print(f"identical {N_NEW} greedy tokens with and without the cache (logits match to 1e-4)")
print(f"total time: no cache {sum(t_nc):.2f}s | with cache {sum(t_c):.2f}s -> {sum(t_nc) / sum(t_c):.1f}x faster")
assert np.median(t_c[-30:]) < 0.5 * np.median(t_nc[-30:]), "late tokens: the cache is much cheaper per step"

In [ ]:
def smooth(x, w=9):
    return np.convolve(x, np.ones(w) / w, mode="valid")

pos = np.arange(32, 32 + N_NEW)
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(pos[4:-4], 1e3 * smooth(t_nc), label="no cache: re-process everything")
ax.plot(pos[5:-4], 1e3 * smooth(t_c[1:]), label="KV cache: one token per step")
ax.scatter([32], [1e3 * t_c[0]], color="tab:red", zorder=3, label="prefill of the 32-token prompt (with cache)")
ax.set_xlabel("sequence length when the token is generated"); ax.set_ylabel("ms per generated token")
ax.set_title("Per-token cost: grows with length without a cache, ~flat with it")
ax.legend(); ax.grid(alpha=0.3)
plt.show()
print(f"no cache: {1e3 * np.median(t_nc[:20]):.1f} ms/token early -> {1e3 * np.median(t_nc[-20:]):.1f} ms/token late")
print(f"cache   : {1e3 * np.median(t_c[1:21]):.1f} ms/token early -> {1e3 * np.median(t_c[-20:]):.1f} ms/token late")

Same tokens, very different cost. Without a cache, the time per token climbs as the sequence grows (the whole prefix is recomputed); with the cache it stays roughly flat (on a GPU it creeps up slowly, because attention still *reads* the growing cache). The price is memory - which is the whole story of LLM serving.

## 3. What the cache costs: the memory formula

For every layer, every KV head and every past token we store one key and one value vector of size head_dim:

`KV bytes = 2 (K and V) x layers x kv_heads x head_dim x seq_len x batch x bytes_per_value`

First check it against the tensors our own model actually holds:

In [ ]:
def kv_cache_bytes(layers, kv_heads, head_dim, seq_len, batch=1, bytes_per=2):
    return 2 * layers * kv_heads * head_dim * seq_len * batch * bytes_per

actual = sum(c["k"].numel() * c["k"].element_size() + c["v"].numel() * c["v"].element_size() for c in caches)
seq_len = out_c.shape[1] - 1                          # the last generated token was never fed back in
formula = kv_cache_bytes(layers=4, kv_heads=4, head_dim=32, seq_len=seq_len, batch=1, bytes_per=4)
print(f"cache tensors: {actual:,} bytes | formula: {formula:,} bytes | shape per layer: {tuple(caches[0]['k'].shape)}")
assert actual == formula

Now an 8B-class model (Llama-3-8B shape: 32 layers, 32 query heads, head_dim 128) in fp16, comparing MHA (32 KV heads), GQA (8, what Llama 3 uses) and MQA (1):

In [ ]:
contexts = {"8k": 8_192, "32k": 32_768, "128k": 131_072}
variants = {"MHA (32 KV heads)": 32, "GQA (8 KV heads)": 8, "MQA (1 KV head)": 1}
rows = []
for name, kvh in variants.items():
    per_tok = kv_cache_bytes(32, kvh, 128, 1)
    rows.append([name, per_tok / 1024] + [kv_cache_bytes(32, kvh, 128, n) / 2**30 for n in contexts.values()])
kv_table = pd.DataFrame(rows, columns=["variant", "KiB/token"] + [f"GiB @ {c}" for c in contexts])
print(kv_table.round(2).to_string(index=False))
assert kv_table.loc[1, "GiB @ 8k"] == 1.0 and kv_table.loc[1, "GiB @ 128k"] == 16.0

x = np.arange(len(contexts)); w = 0.27
fig, ax = plt.subplots(figsize=(9, 3.8))
for j, (name, kvh) in enumerate(variants.items()):
    vals = [kv_cache_bytes(32, kvh, 128, n) / 2**30 for n in contexts.values()]
    ax.bar(x + (j - 1) * w, vals, w, label=name)
ax.axhline(16, color="black", ls="--", lw=1, label="the model weights themselves (8B x 2 bytes ≈ 16 GB)")
ax.set_yscale("log"); ax.set_xticks(x, [f"{c} context" for c in contexts])
ax.set_ylabel("GiB per sequence (log scale)"); ax.set_title("KV cache of an 8B model, one sequence, fp16")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
plt.show()

At 128k context a single MHA sequence would need 64 GiB of cache - four times the weights. GQA-8 brings it to 16 GiB, which is why every modern model uses GQA. And it multiplies by batch size: the cache, not the weights, usually decides how many requests fit on a GPU, and therefore the cost per token. Further levers: FP8/INT8 KV-cache quantisation (halves it again), sliding-window layers, and offloading cold cache to CPU.

In [ ]:
llama70b_8k = kv_cache_bytes(layers=80, kv_heads=8, head_dim=128, seq_len=8_192)
print(f"Llama-3-70B (80 layers, 8 KV heads): {kv_cache_bytes(80, 8, 128, 1) / 1024:.0f} KiB/token, "
      f"{llama70b_8k / 1e9:.2f} GB per 8k sequence, {kv_cache_bytes(80, 8, 128, 4096) / 1e9:.2f} GB per 4k sequence")
print(f"a batch of 32 x 8k sequences: {32 * llama70b_8k / 1e9:.0f} GB of cache on top of ~141 GB of fp16 weights")

## 4. Prefill vs decode, measured: TTFT and TPOT

Prefill cost grows with prompt length (TTFT); each decode step costs about the same (TPOT). Prefill processes hundreds of tokens per forward pass, decode processes one - so prefill tokens/second is far higher.

In [ ]:
def best_of(fn, k=3):
    best = float("inf")
    for _ in range(k):
        t0 = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t0)
    return best

prompt_lens = [16, 64, 128, 256, 512]
prefill_t, decode_t = [], []
with torch.no_grad():
    for n in prompt_lens:
        p = torch.randint(0, 256, (1, n))
        prefill_t.append(best_of(lambda: model(p, [{} for _ in model.blocks])))
        caches_n = [{} for _ in model.blocks]; model(p, caches_n)
        tok = torch.randint(0, 256, (1, 1))
        def one_decode():
            c = [{"k": d["k"], "v": d["v"]} for d in caches_n]          # fresh dicts: do not grow the cache
            model(tok, c, start_pos=n)
        decode_t.append(best_of(one_decode))

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(prompt_lens, np.array(prefill_t) * 1e3, "o-", label="prefill of the whole prompt (≈ TTFT)")
ax.plot(prompt_lens, np.array(decode_t) * 1e3, "s-", label="one decode step at that context (≈ TPOT)")
ax.set_xlabel("prompt length (tokens)"); ax.set_ylabel("ms"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("TTFT grows with the prompt; TPOT barely moves")
plt.show()
tps_prefill, tps_decode = prompt_lens[-1] / prefill_t[-1], 1 / decode_t[-1]
print(f"at 512 tokens: prefill {tps_prefill:,.0f} tokens/s vs decode {tps_decode:,.0f} tokens/s")
assert tps_prefill > 3 * tps_decode, "parallel prefill processes tokens much faster than serial decode"

n_out = 200
latency = prefill_t[3] + n_out * decode_t[3]
print(f"request with a 256-token prompt and {n_out} output tokens: TTFT {prefill_t[3] * 1e3:.1f} ms + "
      f"{n_out} x TPOT {decode_t[3] * 1e3:.2f} ms = {latency:.2f} s total")

| Metric | What it is | Dominated by | Levers |
|---|---|---|---|
| **TTFT** | request sent -> first token | queueing + prefill (compute-bound) | shorter prompts, **prefix/prompt caching**, chunked prefill, faster GPU |
| **TPOT** (inter-token latency) | time between output tokens | reading weights + KV cache each step (memory-bound) | smaller/quantised models, GQA, **speculative decoding**, smaller batch |
| **Throughput** | tokens/s across all users | batch size | bigger batches, continuous batching, PagedAttention |
| **End-to-end latency** | TTFT + TPOT x (output tokens - 1) | output length! | ask for shorter answers, stream the output |

**Why decode is memory-bound (in35):** in a decode step each weight is loaded from GPU memory once and used for only ~2 FLOPs per sequence in the batch (a multiply and an add). A GPU does ~100-300 FLOPs in the time it loads one byte, so with batch 1 it idles waiting for memory. Batching many sequences reuses each loaded weight for many tokens - that is the next section.

## 5. Batching: why throughput and latency fight

In [ ]:
batch_sizes = [1, 2, 4, 8, 16, 32]
step_times = []
with torch.no_grad():
    for bsz in batch_sizes:
        p = torch.randint(0, 256, (bsz, 128))
        cb = [{} for _ in model.blocks]; model(p, cb)
        tok = torch.randint(0, 256, (bsz, 1))
        step_times.append(best_of(lambda: model(tok, [{"k": d["k"], "v": d["v"]} for d in cb], start_pos=128), k=5))
throughput = np.array(batch_sizes) / np.array(step_times)

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(batch_sizes, throughput, "o-", color="tab:green", label="throughput (tokens/s, all sequences)")
ax.set_xscale("log", base=2); ax.set_xticks(batch_sizes, batch_sizes); ax.set_xlabel("batch size (sequences decoding together)")
ax.set_ylabel("tokens / s"); ax2 = ax.twinx()
ax2.plot(batch_sizes, np.array(step_times) * 1e3, "s--", color="tab:red", label="latency per step (ms)")
ax2.set_ylabel("ms per step")
ax.set_title("Batching multiplies throughput; each user's step gets slower")
ax.legend(loc="upper left"); ax2.legend(loc="lower right")
plt.show()
print("throughput x vs batch 1:", {b: round(float(v), 1) for b, v in zip(batch_sizes, throughput / throughput[0])})
assert throughput[-1] > 3 * throughput[0]

One decode step for 32 sequences costs only a few times more than for one sequence (the weights are read once for all of them), so throughput rises steeply while each user's per-token latency rises slowly - until memory (the KV cache!) or compute runs out. Choosing the batch size is choosing a point on that latency/throughput curve.

### Static vs continuous batching

Real requests have very different output lengths. **Static batching** fills a batch and waits until its *longest* request finishes; short requests leave their slot idle. **Continuous (in-flight) batching** - vLLM, TGI, TensorRT-LLM - re-schedules at every decode step: a finished request leaves, a queued one joins immediately.

In [ ]:
def simulate(lengths, slots, continuous):
    """Returns total decode steps, slot utilisation, and a timeline [(request, slot, start, end)]."""
    queue, t, timeline = list(range(len(lengths))), 0, []
    if not continuous:
        while queue:
            batch, queue = queue[:slots], queue[slots:]
            for s, r in enumerate(batch):
                timeline.append((r, s, t, t + lengths[r]))
            t += max(lengths[r] for r in batch)             # everyone waits for the longest
    else:
        free_at = [0] * slots
        for r in queue:
            s = int(np.argmin(free_at))                     # the first slot to become free
            timeline.append((r, s, free_at[s], free_at[s] + lengths[r]))
            free_at[s] += lengths[r]
        t = max(free_at)
    busy = sum(lengths)
    return t, busy / (t * slots), timeline

lengths = rng.integers(10, 200, size=16).tolist()           # output tokens per request
res = {name: simulate(lengths, slots=4, continuous=c) for name, c in [("static", False), ("continuous", True)]}
fig, axes = plt.subplots(2, 1, figsize=(10, 4.2), sharex=True)
for ax, (name, (total, util, tl)) in zip(axes, res.items()):
    for r, s, a, b in tl:
        ax.barh(s, b - a, left=a, color=plt.cm.tab20(r % 20), edgecolor="white")
        ax.text(a + (b - a) / 2, s, str(r), ha="center", va="center", fontsize=7)
    ax.set_yticks(range(4), [f"slot {i}" for i in range(4)])
    ax.set_title(f"{name} batching: {total} steps, slot utilisation {util:.0%}", fontsize=10)
axes[1].set_xlabel("decode step")
plt.tight_layout(); plt.show()
assert res["continuous"][0] < res["static"][0]
print(f"continuous batching finishes the same 16 requests in {res['continuous'][0] / res['static'][0]:.0%} of the steps")

## 6. PagedAttention and prefix caching

Pre-allocating a contiguous cache for the *maximum* length of every request wastes most of GPU memory (internal fragmentation), which caps the batch size. **PagedAttention** (vLLM) stores the cache in fixed-size blocks (e.g. 16 tokens) like OS virtual-memory pages; a per-sequence *block table* maps logical blocks to physical ones, so memory is allocated on demand and at most one partly-filled block per sequence is wasted.

```
seq A logical blocks:  [A0][A1][A2]          block table A: 0->7, 1->2, 2->9
seq B logical blocks:  [B0][B1]              block table B: 0->4, 1->11
physical GPU blocks :  0 1 [A1] 3 [B0] 5 6 [A0] 8 [A2] 10 [B1] ...   (any free block, no contiguity needed)
```

In [ ]:
MAX_LEN, BLOCK = 2048, 16
actual_lens = rng.integers(50, 1200, size=100)                           # tokens each request really uses
reserved_contig = len(actual_lens) * MAX_LEN
reserved_paged = int(np.sum(np.ceil(actual_lens / BLOCK) * BLOCK))
used = int(actual_lens.sum())
print(f"contiguous max-length reservation: {100 * (1 - used / reserved_contig):.0f}% of reserved KV memory wasted")
print(f"paged (16-token blocks)          : {100 * (1 - used / reserved_paged):.1f}% wasted")
print(f"-> the same memory holds {reserved_contig / reserved_paged:.1f}x more sequences")
assert 1 - used / reserved_paged < 0.05 < 0.5 < 1 - used / reserved_contig

Blocks also make **prefix caching** cheap: if many requests start with the same system prompt (or the same document), the blocks for that prefix are identified by a hash of their tokens and *shared*, so their prefill is skipped and their memory is stored once. That is the mechanism behind vLLM's automatic prefix caching and the providers' discounted "cached input tokens" (it is also why a timestamp at the top of a prompt kills your cache hit rate).

In [ ]:
class PrefixCache:
    """Maps each full block of tokens (identified by its whole prefix) to a physical block id."""
    def __init__(self):
        self.blocks, self.allocated, self.hits = {}, 0, 0
    def allocate(self, tokens):
        ids = []
        for i in range(0, len(tokens), BLOCK):
            key = hash(tuple(tokens[: i + BLOCK]))          # a block is reusable only if everything before it matches
            if key in self.blocks and i + BLOCK <= len(tokens):
                self.hits += 1
            else:
                self.blocks[key] = self.allocated; self.allocated += 1
            ids.append(self.blocks[key])
        return ids

system_prompt = list(range(512))                            # 512 shared tokens = 32 blocks
pc = PrefixCache()
for user in range(10):
    question = list(rng.integers(1000, 2000, size=40))
    pc.allocate(system_prompt + question)
no_sharing = 10 * int(np.ceil((512 + 40) / BLOCK))
print(f"10 requests: {pc.allocated} physical blocks with prefix sharing vs {no_sharing} without; "
      f"{pc.hits} block hits = {pc.hits * BLOCK:,} prompt tokens whose prefill was skipped")
assert pc.allocated < no_sharing / 3

## 7. Speculative decoding

Decode is memory-bound, so verifying *several* tokens in one forward pass costs about the same as generating one. Speculative decoding exploits that: a small **draft** model proposes γ tokens cheaply; the big **target** model scores all of them in one pass and keeps the longest correct prefix. A rejection-sampling rule makes the output distribution **exactly** the target's:

- draft proposes x ~ q; accept with probability min(1, p(x) / q(x));
- if rejected, sample from the residual `max(0, p - q)` (renormalised).

Let us verify the "exactly the target's distribution" claim empirically:

In [ ]:
p_target = np.array([0.35, 0.25, 0.15, 0.10, 0.07, 0.05, 0.02, 0.01])
q_draft = np.array([0.20, 0.30, 0.20, 0.05, 0.10, 0.05, 0.05, 0.05])     # a decent but imperfect draft

def speculative_step(p, q, r):
    x = r.choice(len(q), p=q)
    if r.random() < min(1.0, p[x] / q[x]):
        return x, True
    residual = np.maximum(p - q, 0)
    return r.choice(len(p), p=residual / residual.sum()), False

r = np.random.default_rng(1)
draws = [speculative_step(p_target, q_draft, r) for _ in range(20_000)]
freq = np.bincount([x for x, _ in draws], minlength=len(p_target)) / len(draws)
alpha = np.mean([acc for _, acc in draws])
tv = 0.5 * np.abs(freq - p_target).sum()
print("target   :", p_target)
print("empirical:", freq.round(3))
print(f"total-variation distance {tv:.4f} | acceptance rate {alpha:.3f} (theory: sum(min(p, q)) = {np.minimum(p_target, q_draft).sum():.3f})")
assert tv < 0.02 and abs(alpha - np.minimum(p_target, q_draft).sum()) < 0.02

In [ ]:
alphas = np.linspace(0.3, 0.95, 50)
draft_cost = 0.1                                             # one draft step = 10% of a target forward pass
fig, ax = plt.subplots(figsize=(8, 3.6))
for gamma in [2, 4, 8]:
    expected = (1 - alphas ** (gamma + 1)) / (1 - alphas)     # tokens produced per target forward pass
    ax.plot(alphas, expected / (1 + gamma * draft_cost), label=f"γ = {gamma} draft tokens")
ax.axhline(1, color="gray", ls=":", label="no speed-up")
ax.set_xlabel("acceptance rate α"); ax.set_ylabel("speed-up vs plain decoding")
ax.set_title("Speculative decoding pays off only when the draft is usually right")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

With α≈0.8 and γ=4 each target pass yields ~3.4 tokens, a ~2.4x speed-up after paying for the draft; with a poor draft (α≈0.4) long speculation *loses* (wasted draft work). It also loses at large batch sizes, where decode is no longer memory-bound. Variants without a second model: Medusa heads, EAGLE, n-gram / prompt-lookup decoding (great for editing and RAG, where the output copies the input).

## 8. Summary: which trick moves which number

| Technique | What it does | TTFT | TPOT | Throughput | Memory |
|---|---|---|---|---|---|
| KV cache | store past K,V instead of recomputing | - | much better | better | costs a lot |
| GQA / MQA | fewer K/V heads | - | better | better (bigger batch) | 4-32x less cache |
| KV-cache quantisation (FP8/INT8) | smaller cache entries | - | slightly better | better | 2x less |
| Continuous batching | refill slots every step | better (less queueing) | slightly worse | much better | - |
| PagedAttention | block-allocated cache | - | - | much better (bigger batch) | ~no waste |
| Prefix caching | reuse K,V of shared prompt prefixes | much better | - | better | shared |
| Chunked prefill | split long prefills, interleave with decodes | slightly worse for that request | smoother for others | better | - |
| Speculative decoding | draft + verify | - | 2-3x better (if α high) | worse at large batch | + draft model |
| Streaming | send tokens as produced | *perceived* much better | - | - | - |

## Try it yourself

**1.** Use `kv_cache_bytes` to find the maximum batch size of 32k-token sequences for Llama-3-8B (GQA-8, fp16) on an 80 GB GPU, after 16 GB of weights and 10% headroom.

In [ ]:
# Solution — try it yourself first, then run this
free = 80e9 * 0.9 - 16e9
per_seq = kv_cache_bytes(32, 8, 128, 32_768)
print(f"per 32k sequence: {per_seq / 1e9:.2f} GB -> max batch {int(free // per_seq)} "
      f"(with MHA it would be {int(free // kv_cache_bytes(32, 32, 128, 32_768))})")

**2.** In the static-vs-continuous simulation, make every request the same length (100 tokens). Does continuous batching still help? Why?

In [ ]:
# Solution — try it yourself first, then run this
same = [100] * 16
s_steps, s_util, _ = simulate(same, 4, continuous=False)
c_steps, c_util, _ = simulate(same, 4, continuous=True)
print(f"equal lengths: static {s_steps} steps ({s_util:.0%}), continuous {c_steps} steps ({c_util:.0%})")
assert s_steps == c_steps, "with identical lengths nobody waits, so there is nothing to win"

**3.** Speculative decoding with a *perfect* draft (q = p) and with a weak one (uniform q). What acceptance rates do you measure, and what does the speed-up curve predict for γ=4?

In [ ]:
# Solution — try it yourself first, then run this
for name, q in [("perfect draft", p_target), ("weak (uniform) draft", np.full(8, 1 / 8))]:
    rr = np.random.default_rng(2)
    acc = np.mean([speculative_step(p_target, q, rr)[1] for _ in range(5000)])
    e = (1 - acc ** 5) / (1 - acc) if acc < 1 else 5.0
    print(f"{name:14s}: acceptance {acc:.2f} -> speed-up {e / (1 + 4 * draft_cost):.2f}x at γ=4")

## Say it in an interview

- **KV cache, 30 seconds:** "In decode, attention needs the keys and values of every previous token. Because of the causal mask they never change, so we cache them and only compute K and V for the new token. It turns O(n²) recomputation into O(n) work per sequence, and it is paid for in memory: 2 x layers x kv_heads x head_dim x seq_len x batch x bytes."
- **Numbers:** Llama-3-8B (GQA-8) fp16 = 128 KiB/token -> 1 GiB per 8k sequence, 16 GiB at 128k; with MHA it would be 4x. Llama-3-70B = 320 KiB/token -> ~2.7 GB per 8k sequence.
- **Prefill vs decode:** prefill is parallel and compute-bound (sets TTFT); decode is serial and memory-bound (sets TPOT). The KV cache does nothing for TTFT - that is what prompt/prefix caching is for.
- **Throughput vs latency:** batching reuses each weight read across sequences, so throughput scales almost linearly until memory or compute saturates, while each user's TPOT slowly worsens.
- **Serving stack:** continuous batching (refill every step), PagedAttention (block-allocated cache, near-zero waste), prefix caching (share system-prompt blocks), chunked prefill (stop long prompts stalling decodes), speculative decoding (lossless 2-3x TPOT when acceptance is high and batch is small).
- **Traps:** "the KV cache speeds up the first token" (no); forgetting batch in the memory formula; quoting parameters instead of KV cache as the limit on concurrency; speculative decoding is lossless only with the proper rejection rule.

## Next

- [07 · Quantization: int8, int4, NF4](07_quantization_int8_int4_nf4.ipynb) - shrink the weights (and the cache).
- [Latency percentiles and cost](../16_production/05_latency_percentiles_p50_p95_and_cost.ipynb) · [Caching: exact and semantic](../16_production/03_caching_exact_and_semantic.ipynb) · [FastAPI serving and streaming](../16_production/04_fastapi_serving_and_streaming.ipynb)
- [Local LLMs with Ollama](../08_llm_apps/05_local_llms_with_ollama.ipynb) - a real server doing all of this.
- Theory: [genai_flow course](../../genai_flow/index.html) (chapter 6 "Making it fast", 18b's KV cache calculator) · [interview bank](../../ai_interview_prep/index.html)